# Cọc Chịu Tải Trọng Ngang — Biểu Đồ Nội Lực & Chuyển Vị\n\nThư viện: **openpile 1.0.2** + **matplotlib**  \nPhương pháp: FEM 1D, đường cong p-y API (1987)  \nDự án: TTHC-HCM 2026-05"

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

from openpile.construct import Pile, SoilProfile, Layer, Model, BoundaryFixation
from openpile.soilmodels import API_sand
from openpile.winkler import winkler

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "figure.dpi": 110,
})

## 1. Thông Số Đầu Vào\n\nChỉnh các giá trị dưới đây để thay đổi cọc / địa tầng / tải trọng."

In [ ]:
# ── CỌC ──────────────────────────────────────────────────────────
PILE_D_m   = 0.840    # đường kính ngoài (m)
PILE_WT_m  = 0.016    # chiều dày thành (m)
PILE_L_m   = 20.0     # chiều dài cọc (m)
PILE_MAT   = "Steel"  # "Steel" hoặc "Concrete"

# ── TẢI TRỌNG tại đỉnh cọc ───────────────────────────────────────
H_kN   = 100.0   # lực ngang (kN) — dương: +Y
M_kNm  = 0.0     # mô men tại đỉnh (kN·m) — 0 = đầu tự do

# ── MỰC NƯỚC NGẦM ────────────────────────────────────────────────
MNN_m  = -1.0    # âm = dưới mặt đất (m)

# ── ĐỊA TẦNG — danh sách các lớp [(tên, z_top, z_bot, gamma, phi)] ─
LAYERS = [
    ("San lap + set mem  Lop 1",   0.0,   -8.0,  17.5, 26.0),
    ("Cat 2a-2c",                  -8.0,  -14.0, 19.0, 30.0),
    ("Set cung Lop 3+5",          -14.0, -20.0, 19.5, 33.0),
]

print(f"Coc: D={PILE_D_m*1000:.0f}mm  t={PILE_WT_m*1000:.0f}mm  L={PILE_L_m}m")
print(f"Tai: H={H_kN} kN  M={M_kNm} kNm")
print(f"MNN: {MNN_m} m")

## 2. Xây Mô Hình & Giải"

In [ ]:
pile = Pile.create_tubular(
    name=f"D{PILE_D_m*1000:.0f}",
    top_elevation=0.0,
    bottom_elevation=-PILE_L_m,
    diameter=PILE_D_m,
    wt=PILE_WT_m,
    material=PILE_MAT,
)

sp = SoilProfile(
    name="Dia tang TTHC",
    top_elevation=0.0,
    water_line=MNN_m,
    layers=[
        Layer(name=name, top=z_top, bottom=z_bot,
              weight=gamma, lateral_model=API_sand(phi=phi, kind="static"))
        for name, z_top, z_bot, gamma, phi in LAYERS
    ],
)

bc  = [BoundaryFixation(elevation=-PILE_L_m, x=True, y=True, z=True)]
mdl = Model(name="lateral", pile=pile, soil=sp, boundary_conditions=bc)
mdl.set_pointload(elevation=0.0, Py=H_kN, Mx=M_kNm if M_kNm else None)

result = winkler(mdl)

# Lấy kết quả
elev  = result.deflection["Elevation [m]"].values
y_mm  = result.deflection["Deflection [m]"].values * 1000      # mm

# forces có 2 dòng mỗi elevation (top/bottom của phần tử) — lọc unique
fdf   = result.forces.drop_duplicates(subset="Elevation [m]", keep="first")
elev_f = fdf["Elevation [m]"].values
V_kN  = fdf["V [kN]"].values
M_kNm_arr = fdf["M [kNm]"].values

print(f"Hội tụ sau:  {result.displacements.shape[0]} nút")
print(f"Chuyển vị đỉnh cọc  y = {y_mm[0]:.3f} mm")
print(f"Lực cắt lớn nhất    V = {np.nanmax(np.abs(V_kN)):.1f} kN")
print(f"Mô men lớn nhất     M = {np.nanmax(np.abs(M_kNm_arr)):.1f} kN·m")

## 3. Biểu Đồ Nội Lực & Chuyển Vị"

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 8), sharey=True)
fig.suptitle(
    f"Coc D={PILE_D_m*1000:.0f}mm  L={PILE_L_m}m  |  H={H_kN} kN  M={M_kNm} kNm",
    fontsize=12, fontweight="bold",
)

# Màu nền phân biệt địa tầng
layer_colors = ["#fff3cd", "#d1e7dd", "#cfe2ff", "#f8d7da"]
for ax in axes:
    for i, (_, z_top, z_bot, _, _) in enumerate(LAYERS):
        ax.axhspan(z_top, z_bot, alpha=0.18, color=layer_colors[i % len(layer_colors)],
                   label=LAYERS[i][0] if ax is axes[0] else "")
    ax.axhline(MNN_m, color="steelblue", linewidth=1, linestyle="--", alpha=0.6)
    ax.set_ylim(-PILE_L_m * 1.02, 0.5)
    ax.invert_yaxis()

# ── Biểu đồ 1: Chuyển vị ngang y (mm) ────────────────────────────
ax = axes[0]
ax.plot(y_mm, elev, "b-o", markersize=3, linewidth=1.5)
ax.fill_betweenx(elev, 0, y_mm, where=y_mm > 0, alpha=0.15, color="blue")
ax.fill_betweenx(elev, 0, y_mm, where=y_mm < 0, alpha=0.15, color="red")
ax.axvline(0, color="k", linewidth=0.8)
ax.set_xlabel("Chuyển vị y (mm)")
ax.set_ylabel("Cao độ (m)")
ax.set_title("Chuyển vị ngang")
ax.annotate(f"y_max = {y_mm[0]:.2f} mm", xy=(y_mm[0], elev[0]),
            xytext=(y_mm[0]*0.4, elev[3]),
            arrowprops=dict(arrowstyle="->", color="navy"), color="navy", fontsize=9)

# ── Biểu đồ 2: Lực cắt V (kN) ────────────────────────────────────
ax = axes[1]
ax.plot(V_kN, elev_f, "g-o", markersize=3, linewidth=1.5)
ax.fill_betweenx(elev_f, 0, V_kN, alpha=0.15, color="green")
ax.axvline(0, color="k", linewidth=0.8)
ax.set_xlabel("Luc cat V (kN)")
ax.set_title("Lực cắt")
V_abs_max = np.nanmax(np.abs(V_kN))
idx_v = np.nanargmax(np.abs(V_kN))
ax.annotate(f"V_max = {V_abs_max:.1f} kN", xy=(V_kN[idx_v], elev_f[idx_v]),
            xytext=(V_kN[idx_v]*0.3, elev_f[idx_v] - 1.5),
            arrowprops=dict(arrowstyle="->", color="darkgreen"), color="darkgreen", fontsize=9)

# ── Biểu đồ 3: Mô men uốn M (kN·m) ──────────────────────────────
ax = axes[2]
ax.plot(M_kNm_arr, elev_f, "r-o", markersize=3, linewidth=1.5)
ax.fill_betweenx(elev_f, 0, M_kNm_arr, alpha=0.15, color="red")
ax.axvline(0, color="k", linewidth=0.8)
ax.set_xlabel("Mo men M (kN.m)")
ax.set_title("Mô men uốn")
M_abs_max = np.nanmax(np.abs(M_kNm_arr))
idx_m = np.nanargmax(np.abs(M_kNm_arr))
ax.annotate(f"M_max = {M_abs_max:.1f} kNm", xy=(M_kNm_arr[idx_m], elev_f[idx_m]),
            xytext=(M_kNm_arr[idx_m]*0.3, elev_f[idx_m] - 1.5),
            arrowprops=dict(arrowstyle="->", color="darkred"), color="darkred", fontsize=9)

# Chú thích địa tầng bên phải
ax_last = axes[0]
for i, (name, z_top, z_bot, gamma, phi) in enumerate(LAYERS):
    z_mid = (z_top + z_bot) / 2
    ax_last.text(-max(abs(y_mm))*0.05, z_mid, name[:15],
                 fontsize=7.5, color="gray", va="center", ha="right")

plt.tight_layout()
plt.show()

## 4. Đường Cong p-y — Kiểm Tra Huy Động Sức Kháng Đất"

In [ ]:
# Vẽ đường cong p-y tại 3 chiều sâu đại diện
# (openpile trả về p-y mobilization qua result.py_mobilization)
try:
    py_mob = result.py_mobilization  # DataFrame: Elevation, y [m], p [kN/m]
    depths_to_plot = [-1.0, -5.0, -10.0]

    fig2, ax2 = plt.subplots(1, 1, figsize=(7, 5))
    for z in depths_to_plot:
        row = py_mob[np.isclose(py_mob["Elevation [m]"], z, atol=0.3)]
        if not row.empty:
            ax2.scatter(row["y [m]"].values * 1000,
                        row["p [kN/m]"].values,
                        label=f"z = {z} m", s=60)

    ax2.set_xlabel("Chuyển vị y (mm)")
    ax2.set_ylabel("Sức kháng đất p (kN/m)")
    ax2.set_title("Huy động đường cong p-y theo chiều sâu")
    ax2.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"py_mobilization: {e}")
    print("Dùng result.plot_lateral_results() thay thế:")
    result.plot_lateral_results()

## 5. Bảng Kết Quả Tóm Tắt"

In [ ]:
import pandas as pd

# Lấy vị trí M_max và V_max
idx_m = np.nanargmax(np.abs(M_kNm_arr))
idx_v = np.nanargmax(np.abs(V_kN))

summary = pd.DataFrame({
    "Đại lượng": [
        "Chuyển vị đỉnh cọc  y_head",
        "Chuyển vị lớn nhất  y_max",
        "Lực cắt lớn nhất    V_max",
        "Chiều sâu  V_max",
        "Mô men uốn lớn nhất M_max",
        "Chiều sâu  M_max",
    ],
    "Giá trị": [
        f"{y_mm[0]:.3f}",
        f"{np.nanmax(np.abs(y_mm)):.3f}",
        f"{np.nanmax(np.abs(V_kN)):.1f}",
        f"{elev_f[idx_v]:.1f}",
        f"{np.nanmax(np.abs(M_kNm_arr)):.1f}",
        f"{elev_f[idx_m]:.1f}",
    ],
    "Đơn vị": ["mm", "mm", "kN", "m", "kN·m", "m"],
})

display(summary.style.set_properties(**{"text-align": "left"}).hide(axis="index"))